# Chapter 7 — Greedy

*Built with [Claude](https://claude.ai) by Anthropic.*

Greedy algorithms commit to the locally optimal choice at each step without reconsidering. They work when the problem has the greedy choice property — a provable guarantee that local optimality leads to global optimality.

In [1]:
from typing import List

def show_intervals(intervals, title=''):
    """Print intervals as text timeline."""
    if title: print(f"  {title}")
    for s, e in intervals:
        bar = ' ' * s + '─' * (e - s)
        print(f"  [{s:>2},{e:>2}]  |{bar}|")

print('  Helpers loaded')

  Helpers loaded


---

## Part 1 — Interval Scheduling

**DS/ML connection:** Scheduling ML training jobs on a GPU without conflicts, merging overlapping event windows, computing non-overlapping time buckets — all interval problems. Pandas `.resample()` is the tool; the greedy algorithm is what runs underneath when you merge or count overlapping windows.

In [2]:
# ── DS/ML PARALLEL: merge overlapping windows with pandas ──
import pandas as pd

events = [(1,3),(2,6),(8,10),(15,18)]
df = pd.DataFrame(events, columns=['start','end'])
df = df.sort_values('start').reset_index(drop=True)

# pandas groupby trick: detect where gaps occur
df['group'] = (df['start'] > df['end'].shift().cummax()).cumsum()
merged = df.groupby('group').agg({'start':'min', 'end':'max'}).values.tolist()
print('  pandas merge result:', merged)

  pandas merge result: [[1, 6], [8, 10], [15, 18]]


### LC 435 — Non-Overlapping Intervals

**Problem:** Given intervals, return the minimum number of intervals to remove to make the rest non-overlapping.

**DS parallel:** Finding the maximum number of non-conflicting GPU training jobs to schedule — minimize removals = maximize kept intervals.

**Key insight:** Sort by end time. Greedily keep the interval that ends earliest (leaves maximum room for future intervals). Every time we skip one, increment removal count.

In [3]:
# ── DS/ML APPROACH: pandas sort + cummax gap detection ──
def eraseOverlapIntervals_pandas(intervals):
    df = pd.DataFrame(intervals, columns=['s','e'])
    df = df.sort_values('e').reset_index(drop=True)
    # Greedily keep intervals where start >= last kept end
    kept_end = float('-inf')
    removals = 0
    for _, row in df.iterrows():
        if row['s'] >= kept_end:
            kept_end = row['e']
        else:
            removals += 1
    return removals

# ── RAW PYTHON (interview answer) ──
def eraseOverlapIntervals(intervals: List[List[int]]) -> int:
    intervals.sort(key=lambda x: x[1])  # sort by end time
    removals = 0
    end = float('-inf')
    for start, finish in intervals:
        if start >= end:    # no overlap — keep it
            end = finish
        else:
            removals += 1   # overlap — remove
    return removals

ivs = [[1,2],[2,3],[3,4],[1,3]]
print(f"  intervals: {ivs}")
print(f"  pandas:  {eraseOverlapIntervals_pandas([x[:] for x in ivs])} removals")
print(f"  greedy:  {eraseOverlapIntervals([x[:] for x in ivs])} removals")

ivs2 = [[1,2],[1,2],[1,2]]
print(f"  intervals: {ivs2}")
print(f"  greedy:  {eraseOverlapIntervals([x[:] for x in ivs2])} removals")

  intervals: [[1, 2], [2, 3], [3, 4], [1, 3]]
  pandas:  1 removals
  greedy:  1 removals
  intervals: [[1, 2], [1, 2], [1, 2]]
  greedy:  2 removals


In [4]:
# ── TRACE: LC 435 ──
def eraseOverlapIntervals_trace(intervals):
    intervals = sorted(intervals, key=lambda x: x[1])
    print(f"  Sorted by end: {intervals}")
    removals = 0
    end = float('-inf')
    print(f"  {'[s,e]':>10}  {'start>=end?':>12}  {'action':>10}  end")
    for start, finish in intervals:
        if start >= end:
            action = 'KEEP'
            end = finish
        else:
            action = 'REMOVE'
            removals += 1
        overlap = start >= (end if action=='KEEP' else end)
        print(f"  [{start},{finish}]  {str(start >= (end if action=='KEEP' else end)):>12}  {action:>10}  {end}")
    print(f"  Total removals: {removals}")

eraseOverlapIntervals_trace([[1,2],[2,3],[3,4],[1,3]])

  Sorted by end: [[1, 2], [2, 3], [1, 3], [3, 4]]
       [s,e]   start>=end?      action  end
  [1,2]         False        KEEP  2
  [2,3]         False        KEEP  3
  [1,3]         False      REMOVE  3
  [3,4]         False        KEEP  4
  Total removals: 1


### LC 56 — Merge Intervals

**Problem:** Given a list of intervals, merge all overlapping ones and return the merged list.

**DS parallel:** Merging overlapping time windows or session spans — `pd.Interval` merging, or collapsing overlapping event ranges in a log.

**Key insight:** Sort by start. Maintain a running merged interval. If the current interval overlaps (start ≤ merged end), extend the end; otherwise start a new interval.

In [5]:
# ── DS/ML APPROACH: pandas cummax gap trick ──
def merge_pandas(intervals):
    df = pd.DataFrame(intervals, columns=['s','e'])
    df = df.sort_values('s').reset_index(drop=True)
    df['group'] = (df['s'] > df['e'].shift().cummax()).cumsum()
    return df.groupby('group').agg({'s':'min','e':'max'}).values.tolist()

# ── RAW PYTHON (interview answer) ──
def merge(intervals: List[List[int]]) -> List[List[int]]:
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:             # overlaps
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

ivs = [[1,3],[2,6],[8,10],[15,18]]
print(f"  intervals: {ivs}")
print(f"  pandas: {merge_pandas([x[:] for x in ivs])}")
print(f"  greedy: {merge([x[:] for x in ivs])}")

  intervals: [[1, 3], [2, 6], [8, 10], [15, 18]]
  pandas: [[1, 6], [8, 10], [15, 18]]
  greedy: [[1, 6], [8, 10], [15, 18]]


In [6]:
# ── TRACE: LC 56 ──
def merge_trace(intervals):
    intervals = sorted(intervals, key=lambda x: x[0])
    print(f"  sorted: {intervals}")
    merged = [intervals[0][:]]
    print(f"  init merged: {merged}")
    for start, end in intervals[1:]:
        last = merged[-1]
        if start <= last[1]:
            new_end = max(last[1], end)
            print(f"  [{start},{end}]: overlaps [{last[0]},{last[1]}] → extend to [{last[0]},{new_end}]")
            merged[-1][1] = new_end
        else:
            print(f"  [{start},{end}]: no overlap → new interval")
            merged.append([start, end])
    print(f"  result: {merged}")

merge_trace([[1,3],[2,6],[8,10],[15,18]])

  sorted: [[1, 3], [2, 6], [8, 10], [15, 18]]
  init merged: [[1, 3]]
  [2,6]: overlaps [1,3] → extend to [1,6]
  [8,10]: no overlap → new interval
  [15,18]: no overlap → new interval
  result: [[1, 6], [8, 10], [15, 18]]


---

## Part 2 — Greedy on Arrays (Reachability)

**DS/ML connection:** Feasibility checking in sequential pipelines — "given resource budgets at each stage, can processing reach the end?" The jump game is the canonical version of this.

### LC 55 — Jump Game

**Problem:** Given `nums[i]` = max jump length from index `i`, return `True` if you can reach the last index.

**DS parallel:** Can a sequential pipeline reach the final output stage given per-stage resource budgets?

**Key insight:** Track `max_reach` — the farthest index reachable so far. If the current index ever exceeds `max_reach`, return `False`.

In [7]:
import numpy as np

# ── DS/ML APPROACH: cumulative max of (index + jump) ──
def canJump_numpy(nums):
    reachable = np.arange(len(nums)) + np.array(nums)
    max_reach = np.maximum.accumulate(reachable)
    # Can reach end if max_reach[i] >= i for all i before end
    indices = np.arange(len(nums))
    return bool(np.all(max_reach[:-1] >= indices[1:]))

# ── RAW PYTHON (interview answer) ──
def canJump(nums: List[int]) -> bool:
    max_reach = 0
    for i, jump in enumerate(nums):
        if i > max_reach:
            return False
        max_reach = max(max_reach, i + jump)
    return True

tests = [[2,3,1,1,4], [3,2,1,0,4]]
for nums in tests:
    print(f"  {nums}: numpy={canJump_numpy(nums)}  greedy={canJump(nums)}")

  [2, 3, 1, 1, 4]: numpy=True  greedy=True
  [3, 2, 1, 0, 4]: numpy=False  greedy=False


In [8]:
# ── TRACE: LC 55 ──
def canJump_trace(nums):
    max_reach = 0
    print(f"  nums: {nums}")
    print(f"  {'i':>3}  {'jump':>5}  {'i+jump':>7}  {'max_reach':>10}  blocked?")
    for i, jump in enumerate(nums):
        blocked = i > max_reach
        max_reach = max(max_reach, i + jump)
        print(f"  {i:>3}  {jump:>5}  {i+jump:>7}  {max_reach:>10}  {'YES → False' if blocked else 'no'}")
        if blocked:
            return False
    print(f"  → True (reached end)")
    return True

print('Test 1:')
canJump_trace([2,3,1,1,4])
print()
print('Test 2:')
canJump_trace([3,2,1,0,4])

Test 1:
  nums: [2, 3, 1, 1, 4]
    i   jump   i+jump   max_reach  blocked?
    0      2        2           2  no
    1      3        4           4  no
    2      1        3           4  no
    3      1        4           4  no
    4      4        8           8  no
  → True (reached end)

Test 2:
  nums: [3, 2, 1, 0, 4]
    i   jump   i+jump   max_reach  blocked?
    0      3        3           3  no
    1      2        3           3  no
    2      1        3           3  no
    3      0        3           3  no
    4      4        8           8  YES → False


False

### LC 134 — Gas Station

**Problem:** `n` gas stations in a circle. `gas[i]` = gas available, `cost[i]` = gas to reach next station. Find the starting station index that lets you complete the circuit, or return `-1`.

**DS parallel:** Can a sequential data pipeline sustain itself if each stage has a different throughput (gas) and overhead (cost)?

**Key insight:** If total gas ≥ total cost, a solution exists. Scan for where the cumulative surplus goes negative — reset the start to the next station. The last start that didn't cause a deficit is the answer.

In [9]:
# ── DS/ML APPROACH: numpy cumsum ──
def canCompleteCircuit_numpy(gas, cost):
    net = np.array(gas) - np.array(cost)
    if net.sum() < 0:
        return -1
    # The start is the index after the minimum cumsum prefix
    cumsum = np.cumsum(net)
    return int(np.argmin(cumsum) + 1) % len(gas)

# ── RAW PYTHON (interview answer) ──
def canCompleteCircuit(gas: List[int], cost: List[int]) -> int:
    total, tank, start = 0, 0, 0
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff
        tank  += diff
        if tank < 0:        # can't start from current `start`
            start = i + 1   # try starting after this station
            tank = 0
    return start if total >= 0 else -1

gas  = [1,2,3,4,5]
cost = [3,4,5,1,2]
print(f"  gas={gas}")
print(f"  cost={cost}")
print(f"  numpy: {canCompleteCircuit_numpy(gas, cost)}")
print(f"  greedy: {canCompleteCircuit(gas, cost)}")

  gas=[1, 2, 3, 4, 5]
  cost=[3, 4, 5, 1, 2]
  numpy: 3
  greedy: 3


In [10]:
# ── TRACE: LC 134 ──
def canCompleteCircuit_trace(gas, cost):
    total, tank, start = 0, 0, 0
    print(f"  gas={gas}  cost={cost}")
    print(f"  {'i':>3}  {'diff':>5}  {'tank':>6}  {'total':>6}  action")
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff
        tank  += diff
        if tank < 0:
            action = f"tank<0: reset start→{i+1}"
            start = i + 1
            tank = 0
        else:
            action = ''
        print(f"  {i:>3}  {diff:>5}  {tank:>6}  {total:>6}  {action}")
    result = start if total >= 0 else -1
    print(f"  total={total} ({'>=0 → feasible' if total>=0 else '<0 → impossible'})")
    print(f"  start station: {result}")

canCompleteCircuit_trace([1,2,3,4,5], [3,4,5,1,2])

  gas=[1, 2, 3, 4, 5]  cost=[3, 4, 5, 1, 2]
    i   diff    tank   total  action
    0     -2       0      -2  tank<0: reset start→1
    1     -2       0      -4  tank<0: reset start→2
    2     -2       0      -6  tank<0: reset start→3
    3      3       3      -3  
    4      3       6       0  
  total=0 (>=0 → feasible)
  start station: 3
